In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
JD_FILE = '/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/job_description.docx'

In [3]:
!pip install -q transformers accelerate torch python-docx

In [5]:

# ============================================================
# IMPORTS
# ============================================================

import torch
from docx import Document
from transformers import AutoTokenizer, AutoModelForCausalLM

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_NEW_TOKENS = 2000

# ============================================================
# READ DOCX
# ============================================================

def read_docx(file_path):

    doc = Document(file_path)

    text = "\n".join(
        paragraph.text.strip()
        for paragraph in doc.paragraphs
        if paragraph.text.strip()
    )

    return text


# ============================================================
# LOAD JD
# ============================================================

print("Reading JD...")

jd_text = read_docx(JD_FILE)

print(f"Characters Loaded: {len(jd_text)}")


# ============================================================
# LOAD MODEL
# ============================================================

print("Loading Model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model Loaded")


# ============================================================
# PROMPT
# ============================================================

PROMPT = f"""
You are a Senior Talent Acquisition Specialist.

Analyze the following Job Description and create a detailed Markdown summary.

Return ONLY Markdown.

# Role Summary

## Role Location

## Work Model (On-site / Hybrid / Remote)

## Company Size

## Role Details

## Expected Professionality Traits

- Trait

## Role Responsibilities

- Responsibility

## Required Experience Range

### Ideal Range

### Special Cases

## Required Specific Domain Experience

| Domain | Duration |
|----------|----------|

## Qualifiers

### Ideal Skills

### Mandatory Required Skills

### Optionally Required Skills

### Hands-on Experience on Specific Desired Projects

### Hands-on Experience on Specific Project Stage (POC / MVP / Production)

### Desired Project Scale

### Knowledge Base on Specific Theoretical Concepts and Corresponding Hands-on Experience

## Disqualifiers

### Experience Related

### Undesired Skills

### Undesired Candidate Profile Traits

## Expected Notice Period and Tolerance Range

## Number of Days in Office

## Working Style

Job Description:

{jd_text}
"""


# ============================================================
# CREATE CHAT TEMPLATE
# ============================================================

messages = [
    {
        "role": "system",
        "content": "You are an expert recruiter."
    },
    {
        "role": "user",
        "content": PROMPT
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=8192
).to(model.device)

# ============================================================
# GENERATE OUTPUT
# ============================================================

print("Extracting Information...")

outputs = model.generate(
    **inputs,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    temperature=0.0
)
print("response= ",outputs)
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)
print("response= ",response)
# ============================================================
# EXTRACT ASSISTANT RESPONSE
# ============================================================

try:
    markdown_result = response.split("assistant")[-1].strip()
except:
    markdown_result = response

# ============================================================
# SAVE MARKDOWN
# ============================================================
print("markdown_result",markdown_result)
OUTPUT_FILE = "/content/drive/MyDrive/Hackthon/IndiaRuns/India_runs_data_and_ai_challenge/JD_Analysis.md"

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:
    f.write(markdown_result)

# ============================================================
# DISPLAY RESULT
# ============================================================

print("\n")
print("=" * 80)
print(markdown_result)
print("=" * 80)

print(f"\nMarkdown saved to: {OUTPUT_FILE}")

Reading JD...
Characters Loaded: 9564
Loading Model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model Loaded
Extracting Information...
response=  tensor([[151644,   8948,    198,  ...,    624,  73594, 151645]])
response=  system
You are an expert recruiter.
user

You are a Senior Talent Acquisition Specialist.

Analyze the following Job Description and create a detailed Markdown summary.

Return ONLY Markdown.

# Role Summary

## Role Location

## Work Model (On-site / Hybrid / Remote)

## Company Size

## Role Details

## Expected Professionality Traits

- Trait

## Role Responsibilities

- Responsibility

## Required Experience Range

### Ideal Range

### Special Cases

## Required Specific Domain Experience

| Domain | Duration |
|----------|----------|

## Qualifiers

### Ideal Skills

### Mandatory Required Skills

### Optionally Required Skills

### Hands-on Experience on Specific Desired Projects

### Hands-on Experience on Specific Project Stage (POC / MVP / Production)

### Desired Project Scale

### Knowledge Base on Specific Theoretical Concepts and Corresponding Hands